## 1. Setup & Import

## 1. Setup & Import

In [ ]:
import sys
from pathlib import Path
from IPython.display import Video, display
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

# Add golf_pipeline to path
sys.path.insert(0, str(Path.cwd().parent / 'golf_pipeline'))

from video_augmentation import (
    VideoAugmentor,
    process_folder,
    process_video_file,
    read_video,
    generate_multiple_augmented_videos,
    write_video
)

print("✓ Imports successful")

## 2. Configuration

Use the slider to control how many augmented videos to create per source video.

In [ ]:
# ============= CONFIGURE PATHS =============

# Input folder containing videos
INPUT_FOLDER = Path(r"../TDTU-Golf-Pose-v1/Public Test")

# Output folder for augmented videos
OUTPUT_FOLDER = Path(r"../augmented_dataset")

# Random seed for reproducibility
SEED = 42

# ===========================================

print(f"Input folder: {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")

In [ ]:
# Interactive control for number of augmented videos per source video
video_multiplier_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='Videos/source:',
    continuous_update=True,
    style={'description_width': 'initial'}
)

include_original_checkbox = widgets.Checkbox(
    value=True,
    description='Include original video',
    style={'description_width': 'initial'}
)

apply_time_warp_checkbox = widgets.Checkbox(
    value=True,
    description='Apply time warp (temporal distortion)',
    style={'description_width': 'initial'}
)

apply_flicker_checkbox = widgets.Checkbox(
    value=True,
    description='Apply flicker (lighting variations)',
    style={'description_width': 'initial'}
)

display(widgets.VBox([
    widgets.HTML('<h3>Augmentation Settings</h3>'),
    widgets.HTML('<p>Control how many augmented videos to create per source video:</p>'),
    video_multiplier_slider,
    widgets.HTML('<br>'),
    include_original_checkbox,
    apply_time_warp_checkbox,
    apply_flicker_checkbox
]))

def get_num_augmented():
    """Get current slider value."""
    return int(video_multiplier_slider.value)

## 3. Preview Effects

Visualize what augmentation effects look like on sample frames.

In [ ]:
# Load a sample video to preview effects
sample_videos = list(INPUT_FOLDER.rglob('*.mp4')) + list(INPUT_FOLDER.rglob('*.mov'))

if sample_videos:
    SAMPLE_VIDEO = sample_videos[0]
    print(f"Loading sample: {SAMPLE_VIDEO.name}")
    sample_frames, sample_fps, sample_res = read_video(SAMPLE_VIDEO)
    print(f"Loaded {len(sample_frames)} frames at {sample_fps} FPS\n")
    
    # Show original frame
    plt.figure(figsize=(10, 6))
    plt.imshow(sample_frames[30][:, :, ::-1])  # BGR to RGB
    plt.title('Sample Frame (Original)')
    plt.axis('off')
    plt.show()
else:
    print(f"No videos found in {INPUT_FOLDER}")

In [ ]:
# Generate and display multiple augmented versions of the same frame
if 'sample_frames' in locals():
    print("Creating 8 different augmented versions...\n")
    augmentor = VideoAugmentor()
    
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.ravel()
    
    # Show original
    axes[0].imshow(sample_frames[30][:, :, ::-1])
    axes[0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Generate 8 different augmented versions
    for i in range(1, 9):
        # Each augmentation uses a different seed for variety
        augmented = augmentor.augment_sequence_consistently(
            [sample_frames[30]],
            apply_time_warp=False,
            apply_flicker=False,
            seed=SEED + i
        )
        axes[i].imshow(augmented[0][:, :, ::-1])
        axes[i].set_title(f'Augmentation #{i}', fontsize=12)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("Each augmentation applies a different random combination of:")
    print("  ✓ Brightness/Contrast adjustments")
    print("  ✓ Color (Hue/Saturation) variations")
    print("  ✓ Noise and blur effects")
    print("  ✓ Geometric transforms (flip, rotation, scale)")

## 4. Process Single Video (Test)

Test the augmentation on a single video before batch processing.

In [ ]:
# Test on a single video
if sample_videos:
    TEST_VIDEO = sample_videos[0]
    TEST_OUTPUT = Path(r"test_augmented")
    
    print(f"Testing augmentation on: {TEST_VIDEO.name}")
    print(f"Creating {get_num_augmented()} augmented videos...\n")
    
    output_paths = process_video_file(
        video_path=TEST_VIDEO,
        output_dir=TEST_OUTPUT,
        num_augmented=get_num_augmented(),
        include_original=include_original_checkbox.value,
        apply_time_warp=apply_time_warp_checkbox.value,
        apply_flicker=apply_flicker_checkbox.value,
        seed=SEED
    )
    
    print(f"\n✓ Successfully created {len(output_paths)} videos:")
    for path in output_paths:
        print(f"  - {path.name}")
else:
    print(f"No videos found in {INPUT_FOLDER}")

## 5. Compare Original vs Augmented

Watch the videos to see the effects in action.

In [ ]:
# Display original video
if 'TEST_VIDEO' in locals() and TEST_VIDEO.exists():
    print("Original Video:")
    print("="*60)
    display(Video(str(TEST_VIDEO), width=700))

In [ ]:
# Display augmented videos
if 'TEST_OUTPUT' in locals() and TEST_OUTPUT.exists():
    augmented_videos = sorted(TEST_OUTPUT.glob("*.mp4"))
    
    print(f"Showing {len(augmented_videos)} augmented videos:\n")
    
    for i, video_path in enumerate(augmented_videos, 1):
        print(f"\n{'='*60}")
        print(f"[{i}/{len(augmented_videos)}] {video_path.stem}")
        print('='*60)
        display(Video(str(video_path), width=700))

## 6. Batch Process Entire Dataset

Process all videos in the input folder and create augmented versions.

In [ ]:
# Count videos first
if INPUT_FOLDER.exists():
    all_videos = list(INPUT_FOLDER.rglob('*.mp4')) + list(INPUT_FOLDER.rglob('*.mov'))
    num_videos = len(all_videos)
    num_to_create = num_videos * get_num_augmented()
    
    print(f"\n{'='*60}")
    print("BATCH PROCESSING PREVIEW")
    print('='*60)
    print(f"Source videos found: {num_videos}")
    print(f"Augmented videos per source: {get_num_augmented()}")
    print(f"Include original: {include_original_checkbox.value}")
    print(f"Apply time warp: {apply_time_warp_checkbox.value}")
    print(f"Apply flicker: {apply_flicker_checkbox.value}")
    print(f"\nTotal videos to generate: {num_to_create}")
    print(f"Output folder: {OUTPUT_FOLDER}")
    print('='*60)
else:
    print(f"Input folder not found: {INPUT_FOLDER}")

In [ ]:
# BATCH PROCESS ALL VIDEOS
# WARNING: This may take a long time depending on the number of videos!

if INPUT_FOLDER.exists():
    print("Starting batch processing...\n")
    
    results = process_folder(
        input_folder=INPUT_FOLDER,
        output_folder=OUTPUT_FOLDER,
        num_augmented=get_num_augmented(),
        include_original=include_original_checkbox.value,
        apply_time_warp=apply_time_warp_checkbox.value,
        apply_flicker=apply_flicker_checkbox.value,
        recursive=True,
        seed=SEED
    )
    
    print("\n" + "="*60)
    print("PROCESSING COMPLETE!")
    print("="*60)
    print(f"Total videos processed: {len(results)}")
    print(f"Total augmented videos created: {sum(len(v) for v in results.values())}")
    print(f"Output location: {OUTPUT_FOLDER.resolve()}")
else:
    print(f"Input folder not found: {INPUT_FOLDER}")

## 7. Dataset Statistics

In [ ]:
# Analyze the augmented dataset
if OUTPUT_FOLDER.exists():
    all_augmented = list(OUTPUT_FOLDER.rglob("*.mp4"))
    
    print(f"\n{'='*60}")
    print("AUGMENTED DATASET SUMMARY")
    print('='*60)
    print(f"Output folder: {OUTPUT_FOLDER}")
    print(f"Total augmented videos: {len(all_augmented)}")
    
    # Count originals vs augmented
    original_count = len([v for v in all_augmented if 'original' in v.name])
    aug_count = len([v for v in all_augmented if 'aug' in v.name])
    
    print(f"\nBreakdown:")
    print(f"  - Original videos: {original_count}")
    print(f"  - Augmented videos: {aug_count}")
    
    # Calculate total size
    total_size_mb = sum(v.stat().st_size for v in all_augmented) / (1024 * 1024)
    print(f"\nTotal storage used: {total_size_mb:.2f} MB ({total_size_mb/1024:.2f} GB)")
    
    # Count by band
    bands = {}
    for video in all_augmented:
        for band_name in ['Band 1-2', 'Band 2-4', 'Band 4-6', 'Band 6-8', 'Band 8-10']:
            if band_name in str(video):
                bands[band_name] = bands.get(band_name, 0) + 1
                break
    
    if bands:
        print(f"\nVideos per band:")
        for band, count in sorted(bands.items()):
            print(f"  - {band}: {count} videos")
    
    print('='*60)
else:
    print(f"Output folder not found: {OUTPUT_FOLDER}")

## Summary

### How It Works:
1. **Load source video** frames
2. **For each augmented video**:
   - Sample ONE random transformation (combination of effects)
   - Apply this SAME transformation to ALL frames in the video
   - This ensures temporal consistency (no flickering between frames)
3. **Save augmented videos** with unique random effects

### Augmentation Effects:
- **Always applied** (with random parameters per video):
  - Brightness & Contrast (±30%)
  - Hue/Saturation/Value (HSV color space)
  - Gaussian Noise (σ=10-50)
  - Gaussian Blur (kernel 3-7)
  - Horizontal Flip (50% probability)
  - Rotation (±15°)
  - Scale (±20%)

- **Optional** (controlled by checkboxes):
  - Time Warp: Temporal distortion
  - Flicker: Frame-by-frame brightness variation

### Benefits:
- ✓ Increases dataset size by user-defined factor
- ✓ Each augmented video is unique
- ✓ Maintains temporal consistency (realistic motion)
- ✓ Preserves folder structure
- ✓ Reproducible with seed parameter